# Laboratorio Práctico 2 · Sesión 6

**Versión con código, para quien programa.** Reproduce exactamente los mismos números que la hoja de cálculo `Laboratorio_S06.xlsx`: si algo no coincide, algo se calculó mal.

Corre tal cual en Google Colab o en Replit. Solo usa `scikit-learn`, que ya viene instalado en ambos. La idea de fondo es la misma del pizarrón: precisión y recuperación cuando hay una respuesta correcta contra la cual comparar, y kappa cuando lo que se mide es el acuerdo entre dos codificadores.


## Parte 1 · Precisión y recuperación

Le pedimos a la IA que extrajera los requisitos del Artículo 8 de las reglas de operación de un programa (ficticio). El artículo tiene **6 requisitos reales**. La IA devolvió 4 de ellos, **omitió 2** y **agregó 1** que no estaba.

Cada elemento se describe con dos banderas:

- `es_requisito`: 1 si de verdad aparece en el Artículo 8.
- `la_ia_lo_incluyo`: 1 si apareció en la salida de la IA.

Con eso, `scikit-learn` cuenta los aciertos y errores y calcula las tres medidas.


In [1]:
from sklearn.metrics import precision_score, recall_score, f1_score

# (texto, es_requisito, la_ia_lo_incluyo)
elementos = [
    ("Residir en localidad de alta o muy alta marginacion", 1, 1),
    ("Ingreso del hogar bajo la linea de pobreza extrema",   1, 1),
    ("CURP de la persona titular",                           1, 1),
    ("Comprobante de domicilio reciente",                    1, 1),
    ("No recibir otro apoyo alimentario federal",            1, 0),  # la IA lo omitio  -> FN
    ("Persona titular mayor de edad, una por hogar",         1, 0),  # la IA lo omitio  -> FN
    ("Acta de nacimiento de la persona titular",             0, 1),  # la IA lo invento -> FP
]

y_true = [es_req  for _, es_req,  _ in elementos]   # lo que la norma pide
y_pred = [incluyo for _, _, incluyo in elementos]   # lo que la IA entrego

VP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
FP = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
FN = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
print(f"VP = {VP}    FP = {FP} (invento)    FN = {FN} (omitio)")

print(f"Precision    = {precision_score(y_true, y_pred):.3f}")
print(f"Recuperacion = {recall_score(y_true, y_pred):.3f}")
print(f"F1           = {f1_score(y_true, y_pred):.3f}")


VP = 4    FP = 1 (invento)    FN = 2 (omitio)
Precision    = 0.800
Recuperacion = 0.667
F1           = 0.727


La precisión sale de 0.80 (de 5 cosas que entregó, 4 eran correctas) y la recuperación de 0.667 (de 6 que había, encontró 4). El F1 las resume en 0.727. Una recuperación de 0.667 quiere decir que se escaparon 2 requisitos que sí estaban: en una evaluación que se firma, eso pesa aunque la precisión se vea alta.


## Parte 2 · Acuerdo entre codificadores (kappa)

Ahora la tarea no tiene una respuesta correcta: clasificar la polaridad de 20 comentarios ciudadanos (ficticios). Cada comentario lo etiquetó un **evaluador humano (A)** y la **IA (B)**. Medimos si coinciden más allá de lo que coincidirían por azar, con la kappa de Cohen.


In [2]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# (comentario, etiqueta del humano, etiqueta de la IA)
comentarios = [
    ("El apoyo me llego a tiempo este bimestre, sin problemas.",              "Positiva", "Positiva"),
    ("En la ventanilla me atendieron rapido y me explicaron todo.",          "Positiva", "Positiva"),
    ("Gracias al programa pude comprar la despensa del mes.",                 "Positiva", "Positiva"),
    ("El tramite fue mas sencillo de lo que esperaba.",                       "Positiva", "Positiva"),
    ("El modulo nuevo esta bien ubicado y comodo.",                          "Positiva", "Positiva"),
    ("Me resolvieron la duda del pago en una sola visita.",                   "Positiva", "Positiva"),
    ("Llevo tres meses esperando y el apoyo no cae.",                         "Negativa", "Negativa"),
    ("Me mandaron de una ventanilla a otra sin resolver nada.",              "Negativa", "Negativa"),
    ("El sistema estaba caido y perdi toda la manana.",                       "Negativa", "Negativa"),
    ("Nunca contestan el telefono del modulo.",                              "Negativa", "Negativa"),
    ("Me pidieron un documento que no venia en la lista.",                    "Negativa", "Negativa"),
    ("Cambiaron la fecha de pago y nadie aviso.",                            "Negativa", "Negativa"),
    ("El personal fue grosero cuando pregunte por mi tramite.",              "Negativa", "Negativa"),
    ("Me rechazaron sin explicarme el motivo.",                              "Negativa", "Negativa"),
    ("Ya no batalle nada esta vez, todo salio bien.",                        "Positiva", "Negativa"),  # sinceridad que la IA leyo mal
    ("Sin quejas, el pago llego completo.",                                  "Positiva", "Negativa"),  # negacion que la IA leyo mal
    ("Que maravilla, solo tres visitas para que me digan que falta un papel.","Negativa", "Positiva"),  # sarcasmo
    ("Muy amables para decirme que regrese la proxima semana, otra vez.",    "Negativa", "Positiva"),  # sarcasmo cortes
    ("Excelente servicio, si es que algun dia contestan.",                   "Negativa", "Positiva"),  # sarcasmo
    ("Rapidisimos, apenas dos horas formado bajo el sol.",                   "Negativa", "Positiva"),  # sarcasmo
]

A = [a for _, a, _ in comentarios]   # humano
B = [b for _, _, b in comentarios]   # IA

k = cohen_kappa_score(A, B)
print(f"Kappa de Cohen = {k:.3f}")

etiquetas = ["Positiva", "Negativa"]
cm = confusion_matrix(A, B, labels=etiquetas)
print("\nContingencia (fila = humano, columna = IA):")
print("                  IA:Pos  IA:Neg")
for nombre, fila in zip(etiquetas, cm):
    print(f"  Humano:{nombre:<8} {fila[0]:>5} {fila[1]:>7}")

# La misma kappa a mano, para ver de donde sale
n  = len(A)
Po = (cm[0, 0] + cm[1, 1]) / n
Pe = (cm[0].sum() / n) * (cm[:, 0].sum() / n) + (cm[1].sum() / n) * (cm[:, 1].sum() / n)
kappa_manual = (Po - Pe) / (1 - Pe)
print(f"\nPo = {Po:.3f}   Pe = {Pe:.3f}   kappa = {kappa_manual:.3f}")

def interpreta(k):
    if k < 0:      return "Pobre"
    if k <= 0.20:  return "Leve"
    if k <= 0.40:  return "Aceptable"
    if k <= 0.60:  return "Moderado"
    if k <= 0.80:  return "Sustancial"
    return "Casi perfecto"

print("Interpretacion (Landis y Koch):", interpreta(k))


Kappa de Cohen = 0.400

Contingencia (fila = humano, columna = IA):
                  IA:Pos  IA:Neg
  Humano:Positiva     6       2
  Humano:Negativa     4       8

Po = 0.700   Pe = 0.500   kappa = 0.400
Interpretacion (Landis y Koch): Aceptable


El acuerdo bruto fue del 70%, pero la mitad era esperable por azar, así que kappa baja a 0.40: acuerdo apenas aceptable. Los cuatro desacuerdos no son ruido: son comentarios con sarcasmo o cortesía que la IA leyó al revés. Ahí está el límite de clasificar con IA, y por eso se mide el acuerdo antes de confiar en las categorías.

## Cierre

Todo esto fue sobre texto, y a mano. Calcular kappa sobre 20 comentarios toma minutos; sobre miles de registros es la misma línea de código que acaban de correr. Ahí empieza el análisis cuantitativo y los entornos reproducibles del Módulo III.
